# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mathu2112/FlyRank-ML-Internship-Repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

---
My week 4 baseline was a hand-written rule to identify potential CTR opportunities.The rule flagged observations with atleast 100 impressions,an average search position between 1 and 20, and CTR below 0.005.

For week 5 , I chose to use a **Decision Tree classifie**r.A decision tree is appropriate because it can learn decision rules directly from the data and is interpretable; its decision can be expressed as feature thersholds,making it suitable for comparison with my rule based baseline.

I will compare the Decision Tree against Week 4 baseline using the same holdout split and evaluation metrics.The goal is not to use a more complex model for its own sake,but to test whether a data-driven set of learned rules perform better than existing hand-written baseline.

In [27]:
# This cell is for CODE (numbers, a query, a check).

!pip install -q datasets duckdb pyarrow pandas huggingface_hub

from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

# Access dataset
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

print(dataset)

# Select the train split
data = dataset["train"]

# Check some report dates
print(data["report_date"][:10])

# Inspect available columns
print("\nColumns:")
print(data.column_names)

# Check number of rows
print("\nNumber of rows:", len(data))

# Look at the first 5 rows
print("\nFirst 5 rows:")
print(data[:5])

# Use the same 1,000,000-row sample as Week 4
sample = data.select(range(1_000_000))

# Convert the sample to a pandas DataFrame
df = sample.to_pandas()

print("Shape:", df.shape)
df.head()

import numpy as np

# Calculate CTR safely
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

# Recreate the Week 4 baseline rule
df["baseline_prediction"] = (
    (df["gsc_avg_position"] > 0) &
    (df["gsc_avg_position"] <= 20) &
    (df["gsc_impressions"] >= 100) &
    (df["ctr"] < 0.005)
).astype(int)

# Check how many rows are flagged
print("Baseline predictions:")
print(df["baseline_prediction"].value_counts())

print("\nPercentage flagged:")
print(df["baseline_prediction"].value_counts(normalize=True) * 100)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})
[datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27)]

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gs

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

---
I used an 80/20 stratified random train/test split with random_state = 42.The split was stratified by a target_low_ctr so that the proportion of low-CTR and non-low-CTR observations remained almost identical in both the training and testing sets.

I did not use a grouped-by-client or time-awareness split in the initial analysis.The goal was to compare Week 4 rule-based baseline and Decision Tree under the same evaluation conditions.Both models were therefore evaluated on exactly the same held-out test set using the same target and metrics

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Keep rows with valid GSC data and enough impressions
eligible = df[
    (df["gsc_data_available"] == True) &
    (df["gsc_impressions"] >= 100) &
    (df["gsc_impressions"] > 0)
].copy()

# Find the median CTR among eligible rows
ctr_threshold = eligible["ctr"].median()

# Create an independent target
# 1 = below-median CTR
# 0 = median-or-above CTR
eligible["target_low_ctr"] = (
    eligible["ctr"] < ctr_threshold
).astype(int)

print("Number of eligible rows:", len(eligible))
print("Median CTR threshold:", ctr_threshold)

print("\nTarget distribution:")
print(eligible["target_low_ctr"].value_counts())

print("\nTarget percentage:")
print(eligible["target_low_ctr"].value_counts(normalize=True) * 100)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Number of eligible rows: 67650
Median CTR threshold: 0.002702702702702703

Target distribution:
target_low_ctr
0    33832
1    33818
Name: count, dtype: int64

Target percentage:
target_low_ctr
0    50.010347
1    49.989653
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
feature_cols = [
    "gsc_avg_position",
    "gsc_sum_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

X = eligible[feature_cols].copy()
y = eligible["target_low_ctr"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing values:")
print(X.isnull().sum())

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Get the original rows corresponding to the test set
test_data = eligible.loc[X_test.index].copy()

# Apply the Week 4 baseline rule to the test set
baseline_test_pred = (
    (test_data["gsc_avg_position"] > 0) &
    (test_data["gsc_avg_position"] <= 20) &
    (test_data["gsc_impressions"] >= 100) &
    (test_data["ctr"] < 0.005)
).astype(int)

# Evaluate the baseline
baseline_accuracy = accuracy_score(y_test, baseline_test_pred)
baseline_precision = precision_score(
    y_test, baseline_test_pred, zero_division=0
)
baseline_recall = recall_score(
    y_test, baseline_test_pred, zero_division=0
)
baseline_f1 = f1_score(
    y_test, baseline_test_pred, zero_division=0
)

print("WEEK 4 BASELINE RESULTS")
print("-" * 30)
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1 Score:  {baseline_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, baseline_test_pred))

from sklearn.tree import DecisionTreeClassifier

# Create the Decision Tree
tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    random_state=42
)

# Train the model
tree_model.fit(X_train, y_train)

# Make predictions on the SAME test set
tree_test_pred = tree_model.predict(X_test)

print("Decision Tree trained successfully!")

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Evaluate the Decision Tree
tree_accuracy = accuracy_score(y_test, tree_test_pred)
tree_precision = precision_score(y_test, tree_test_pred, zero_division=0)
tree_recall = recall_score(y_test, tree_test_pred, zero_division=0)
tree_f1 = f1_score(y_test, tree_test_pred, zero_division=0)

print("DECISION TREE RESULTS")
print("-" * 30)
print(f"Accuracy:  {tree_accuracy:.4f}")
print(f"Precision: {tree_precision:.4f}")
print(f"Recall:    {tree_recall:.4f}")
print(f"F1 Score:  {tree_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, tree_test_pred))

import pandas as pd

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": tree_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance


# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Feature shape: (67650, 14)
Target shape: (67650,)

Missing values:
gsc_avg_position            0
gsc_sum_position            0
ga4_pageviews               0
ga4_sessions                0
ga4_users                   0
ga4_engaged_sessions        0
ga4_total_engagement_sec    0
sessions_organic            0
sessions_direct             0
sessions_referral           0
sessions_social             0
sessions_paid               0
sessions_ai                 0
scroll_events               0
dtype: int64
Training set: (54120, 14)
Test set: (13530, 14)

Training target distribution:
target_low_ctr
0    0.500111
1    0.499889
Name: proportion, dtype: float64

Test target distribution:
target_low_ctr
0    0.500074
1    0.499926
Name: proportion, dtype: float64
WEEK 4 BASELINE RESULTS
------------------------------
Accuracy:  0.7841
Precision: 0.8039
Recall:    0.7515
F1 Score:  0.7768

Confusion Matrix:
[[5526 1240]
 [1681 5083]]
Decision Tree trained successfully!
DECISION TREE RESULTS
-----------

,feature,importance
0,gsc_avg_position,0.940506
1,gsc_sum_position,0.059494
2,ga4_pageviews,0.000000
3,ga4_sessions,0.000000
4,ga4_users,0.000000
5,ga4_engaged_sessions,0.000000
6,ga4_total_engagement_sec,0.000000
7,sessions_organic,0.000000
8,sessions_direct,0.000000
9,sessions_referral,0.000000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

---

The largest error group was false negatives,with 2,516 cases.These cases had a relatively good average search position of 6.35. Since the model relied heavily on gsc_avg_position,it appears that search position alone was not sufficient to identify all low-CTR cases.Some content ranked well but still had low CTR, which the model failed to detect using the available non-leaking features.

---

**Observation of Final Comparison table**

The Week 4 rule-based baseline outperformed the Decision Tree on the same test split. The baseline achieved an F1 score of 0.7768, compared with 0.6480 for the Decision Tree. This shows that the more complex machine learning approach did not outperform the existing hand-written rule for this task.

Feature importance showed that the Decision Tree relied almost entirely on gsc_avg_position (94.1%), with gsc_sum_position contributing the remaining 5.9%. The other selected engagement and traffic features were not used by the fitted tree.

Error analysis showed that false negatives were the largest error group. These cases had a mean search position of 6.35, suggesting that some well-ranked content still had low CTR. This helps explain why search-position-based signals alone were insufficient to outperform the Week 4 rule.

In [30]:
comparison = pd.DataFrame({
    "Model": [
        "Week 4 Rule Baseline",
        "Decision Tree (depth=5)"
    ],
    "Accuracy": [
        baseline_accuracy,
        tree_accuracy
    ],
    "Precision": [
        baseline_precision,
        tree_precision
    ],
    "Recall": [
        baseline_recall,
        tree_recall
    ],
    "F1 Score": [
        baseline_f1,
        tree_f1
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Week 4 Rule Baseline,0.784109,0.803891,0.751478,0.776801
1,Decision Tree (depth=5),0.658906,0.669293,0.628031,0.648005


## Self-check

Before you submit, confirm each line honestly:

- [Yes] Every section above is filled — markdown thinking AND the code that backs it
- [Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes] No client names, URLs, or private queries anywhere
- [Yes ] My claims use careful words: observed, measured, directional, decision-support
- [Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.